# Lab 07 — On-Policy Distillation: The Run You Babysit

**Tier 2 lab.** Part A executes and asserts anywhere; Part B trains only when
`RUN_TRAINING = True`.

**The question.** Every lab so far trained the student on *someone else's* token sequences —
the corpus's, or the teacher's. But at inference the student conditions on **its own** past
tokens, including its own mistakes, which teacher-forced training never shows it. That mismatch
is **exposure bias**, and its signature is a student that starts a sentence well and degrades
as its own errors compound.

GKD's fix (Agarwal et al., 2306.13649) is to train on the student's own rollouts: the student
*generates*, the teacher *scores* the generated tokens, and the divergence is minimized on the
student's actual state distribution. Two knobs define the whole method, and this lab sweeps the
one Lab 05 didn't:

- **`lmbda`** — the fraction of training data that is student-generated. 0 = pure off-policy
  (Lab 03), 1 = pure on-policy, 0.5 = mixed.
- **`beta`** — the divergence, exactly Lab 05's knob, typically pushed toward reverse KL
  on-policy because scoring the student's own samples is where mode-seeking is safest.

**The economics, one more time, because they are widely gotten backwards** (this course's own
README once got them backwards and says so): on-policy looks expensive because "generation is
in the loop" — but *the student generates*, and the student is the small model. The teacher
only ever runs prefill over already-generated tokens. On bandwidth-limited hardware this is
the *cheap* configuration. The genuinely expensive one was Lab 06's, where the big model
decoded.

**What is genuinely new here is risk.** Training on self-generated data creates a feedback
loop: the student narrows, its rollouts narrow, the objective rewards further narrowing.
Left alone this is **entropy collapse**, then length collapse, then a model that emits the
same three phrases forever. On-policy runs are therefore *babysat* runs — and Part A builds
and tests the babysitter before Part B needs it.

In [1]:
import sys, os, json, math, dataclasses
sys.path.insert(0, "../code")

import torch
import torch.nn.functional as F

from kd_core import mean_entropy, distinct_n, onpolicy_mask
from kd_pipeline import set_seed_everywhere, config_fingerprint, EntropyMonitor, RunManifest

RUN_TRAINING = False        # <-- flip on the training box
SEED = 17
set_seed_everywhere(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"torch {torch.__version__} | device: {device} | RUN_TRAINING: {RUN_TRAINING}")

torch 2.13.0+cpu | device: cpu | RUN_TRAINING: False


## Part A · 1 — The trainer's contract, introspected live

Two TRL trainers implement on-policy distillation, and they are not interchangeable:

- `trl.experimental.gkd.GKDTrainer` — the GKD paper's method: `lmbda` mixes on- and off-policy
  batches, `beta` picks the divergence, `seq_kd` degrades it to Lab 06.
- `trl.experimental.distillation.DistillationTrainer` — the newer path: *always* on-policy
  (no `lmbda` at all), teacher servable over vLLM, Liger-fused JSD.

Both live under `trl.experimental`, which means the API can change under you between minor
versions — this course's README once described a field that no longer exists. Hence the
standing rule, enforced here as executable pre-flight: **introspect the installed objects for
every field your run plan depends on.** A failed assertion here costs one second; discovering
the same drift from inside a wedged training run costs an afternoon.

In [2]:
from trl.experimental.gkd import GKDConfig
from trl.experimental.distillation import DistillationConfig

gkd = {f.name: f.default for f in dataclasses.fields(GKDConfig)}
dis = {f.name for f in dataclasses.fields(DistillationConfig)}

for k in ("lmbda", "beta", "temperature", "max_new_tokens", "seq_kd"):
    assert k in gkd, f"TRL drift: GKDConfig lost '{k}' — re-ground before running"
    print(f"GKDConfig.{k:<16} default = {gkd[k]}")

assert "beta" in dis and "lmbda" not in dis, \
    "DistillationTrainer contract changed: it is documented as always-on-policy (no lmbda)"
print("\nDistillationConfig: beta present, lmbda absent -> always on-policy, as documented.")
print("Both trainers verified against the installed TRL, not against memory of the docs.")

/tmp/ipykernel_7565/872399649.py:1: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.gkd import GKDConfig
/usr/local/lib/python3.11/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


GKDConfig.lmbda            default = 0.5
GKDConfig.beta             default = 0.5
GKDConfig.temperature      default = 0.9
GKDConfig.max_new_tokens   default = 128
GKDConfig.seq_kd           default = False

DistillationConfig: beta present, lmbda absent -> always on-policy, as documented.
Both trainers verified against the installed TRL, not against memory of the docs.


## Part A · 2 — Build the babysitter, then prove it on synthetic runs

`kd_pipeline.EntropyMonitor` is the tripwire: it flags collapse when rollout entropy crosses an
absolute floor or loses a large fraction of its value inside a trailing window. Simple by
design — the goal is not a perfect detector but *any* automatic one, because the human watching
a loss curve reliably notices collapse a few hundred steps after it began, and by then the
checkpoint worth keeping is gone.

Test it like any detector: feed it a healthy trajectory (entropy declining gently toward a
plateau — every successful distillation does this, decline alone is NOT collapse), a collapsing
one, and a floor-crossing one. Assert it stays quiet on the first and fires on the others.
A monitor you have not seen fire is a monitor you do not have.

In [3]:
healthy = EntropyMonitor(floor_nats=0.15, drop_frac=0.6, window=20)
for step in range(0, 2000, 50):
    healthy.update(step, 2.5 * math.exp(-step / 1500) + 1.2)   # 3.7 -> ~1.5, plateauing
assert not healthy.collapsed, "gentle decline to a plateau must NOT trip the monitor"
print("healthy run:  ", healthy.report())

collapsing = EntropyMonitor(floor_nats=0.15, drop_frac=0.6, window=20)
for step in range(0, 2000, 50):
    h = 2.5 if step < 1200 else 2.5 * math.exp(-(step - 1200) / 120)
    collapsing.update(step, h + 0.05)
assert collapsing.collapsed, "a fast late-run drop must trip the window rule"
print("collapsing run:", collapsing.report())

floor = EntropyMonitor(floor_nats=0.15)
for step, h in [(0, 1.0), (100, 0.5), (200, 0.10)]:
    floor.update(step, h)
assert floor.collapsed, "entropy under the absolute floor must trip"
print("floor-crossing:", floor.report())
print("\nthe babysitter fires when it should and stays quiet when it should")

healthy run:   entropy 3.700 nats @ step 0 -> 1.881 @ step 1950 (healthy)
collapsing run: entropy 2.550 nats @ step 0 -> 0.055 @ step 1950 (COLLAPSED)
floor-crossing: entropy 1.000 nats @ step 0 -> 0.100 @ step 200 (COLLAPSED)

the babysitter fires when it should and stays quiet when it should


## Part A · 3 — The arms, and the mask that guards the rollouts

Five runs. The lmbda sweep isolates the on-policy fraction; the cold-start pair isolates
initialization; the fifth arm is **deliberately misconfigured** so that you see a collapse
happen under supervision before one ever surprises you:

| arm | lmbda | beta | init | purpose |
|---|---|---|---|---|
| `off` | 0.0 | 0.5 | base 360M | pure off-policy reference |
| `mixed` | 0.5 | 0.5 | base 360M | the GKD default |
| `on` | 1.0 | 0.5 | base 360M | pure on-policy |
| `on-warm` | 1.0 | 0.5 | **Lab 04's distilled student** | cold start vs warm start |
| `degenerate` | 1.0 | **1.0** | base 360M | reverse-KL + low sampling T + hot lr: watch it collapse |

The cold-start logic deserves one paragraph, because it is Lab 00 §9 wearing production
clothes: on-policy scoring is a *sampled-ratio* regime, trustworthy when student and teacher
roughly agree. A base student disagrees everywhere, so its early rollouts are garbage scored
noisily — the off-policy cold start problem. Warm-starting from Lab 04's checkpoint spends
cheap cached-logit steps to buy a student close enough for on-policy steps to mean something.
That is also the course's answer to "off-policy or on-policy?": *sequence them*.

One mechanical trap before Part B: rollouts end at EOS, and everything after EOS in the padded
batch is garbage the loss must not see. `kd_core.onpolicy_mask` exists for exactly this;
asserted here on a hand-built batch, including the never-emitted-EOS row (masked entirely
open — which is why generation caps matter).

In [4]:
ARMS = {
    "off":        dict(lmbda=0.0, beta=0.5, init="base",  lr=3e-5, gen_T=0.9),
    "mixed":      dict(lmbda=0.5, beta=0.5, init="base",  lr=3e-5, gen_T=0.9),
    "on":         dict(lmbda=1.0, beta=0.5, init="base",  lr=3e-5, gen_T=0.9),
    "on-warm":    dict(lmbda=1.0, beta=0.5, init="lab04", lr=3e-5, gen_T=0.9),
    "degenerate": dict(lmbda=1.0, beta=1.0, init="base",  lr=2e-4, gen_T=0.3),
}
def diff_keys(a, b):
    return {k for k in a if a[k] != b[k]}
assert diff_keys(ARMS["off"], ARMS["on"]) == {"lmbda"}
assert diff_keys(ARMS["mixed"], ARMS["on"]) == {"lmbda"}
assert diff_keys(ARMS["on"], ARMS["on-warm"]) == {"init"}
assert len(diff_keys(ARMS["on"], ARMS["degenerate"])) == 3, "degenerate is aggressive on purpose"

# onpolicy_mask: supervise up to and including first EOS, nothing after.
EOS = 7
gen = torch.tensor([[4, 5, EOS, 9, 9],      # normal: mask [1,1,1,0,0]
                    [4, 5, 6, 8, EOS],      # EOS at the end: all supervised
                    [4, 5, 6, 8, 9]])       # never stopped: all supervised (cap risk!)
m = onpolicy_mask(gen, eos_token_id=EOS)
assert m.tolist() == [[True, True, True, False, False],
                     [True, True, True, True, True],
                     [True, True, True, True, True]]
print("arm discipline + rollout masking verified")
print("row 3 is the cautionary one: a capped, EOS-less rollout is fully supervised —")
print("filter those or cap generously, or you train a student that never stops.")

arm discipline + rollout masking verified
row 3 is the cautionary one: a capped, EOS-less rollout is fully supervised —
filter those or cap generously, or you train a student that never stops.


## Part B — Five babysat runs

`GKDTrainer` owns the loop; the course attaches its own eyes via a `TrainerCallback` that, on
every logging step, samples a small batch of rollouts from the *current* student, updates the
`EntropyMonitor`, prints two live generations (read them — mid-run text tells you things
metrics cannot), and **stops the run** via `control.should_training_stop` when the monitor
trips. The degenerate arm exists so you can watch that whole chain fire for real: entropy
sliding, samples going repetitive, tripwire, halt — with the healthy arms as the contrast.

In [5]:
from transformers import (AutoModelForCausalLM, AutoTokenizer, TrainerCallback)
from trl.experimental.gkd import GKDConfig, GKDTrainer
from datasets import Dataset
import glob

class RolloutWatch(TrainerCallback):
    def __init__(self, tok, prompts, monitor, gen_T, every=50):
        self.tok, self.prompts, self.monitor = tok, prompts, monitor
        self.gen_T, self.every = gen_T, every

    def on_log(self, args, state, control, model=None, **kw):
        if model is None or state.global_step % self.every:
            return
        model.eval()
        with torch.no_grad():
            enc = self.tok(self.prompts, return_tensors="pt", padding=True,
                           padding_side="left").to(model.device)
            gen = model.generate(**enc, do_sample=True, temperature=self.gen_T,
                                 max_new_tokens=64, pad_token_id=self.tok.eos_token_id)
            new = gen[:, enc["input_ids"].shape[1]:]
            logits = model(gen).logits[:, enc["input_ids"].shape[1]-1:-1]
            m = onpolicy_mask(new, self.tok.eos_token_id)
            H = mean_entropy(logits.float(), m)
        model.train()
        self.monitor.update(state.global_step, H)
        texts = [self.tok.decode(new[i], skip_special_tokens=True) for i in range(2)]
        print(f"  step {state.global_step}: rollout entropy {H:.3f} | {self.monitor.report()}")
        for t in texts:
            print(f"    sample: {t[:110]!r}")
        if self.monitor.collapsed:
            print("  !! ENTROPY COLLAPSE — stopping this arm, keeping the last checkpoint")
            control.should_training_stop = True

def run_arm(name, arm, base_cfg):
    set_seed_everywhere(SEED)
    tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-360M-Instruct")
    init = ("HuggingFaceTB/SmolLM2-360M-Instruct" if arm["init"] == "base"
            else sorted(glob.glob("../runs/lab04/cached_*"))[-1])
    student = AutoModelForCausalLM.from_pretrained(init, dtype=torch.bfloat16)
    teacher = AutoModelForCausalLM.from_pretrained(
        "HuggingFaceTB/SmolLM2-1.7B-Instruct", dtype=torch.bfloat16)

    tr = torch.load("../data/lab03/train.pt")     # prompts come from the shared corpus
    msgs = []
    for i in range(1024):
        text = tok.decode(tr["input_ids"][i, :tr["prompt_lens"][i]], skip_special_tokens=True)
        msgs.append({"messages": [{"role": "user", "content": text},
                                  {"role": "assistant", "content": ""}]})
    ds = Dataset.from_list(msgs)

    args = GKDConfig(
        output_dir=f"../runs/lab07/{name}",
        lmbda=arm["lmbda"], beta=arm["beta"], temperature=arm["gen_T"],
        max_new_tokens=128, per_device_train_batch_size=4,
        gradient_accumulation_steps=8, learning_rate=arm["lr"],
        max_steps=600, logging_steps=25, save_steps=200, bf16=True,
        report_to=[],
    )
    monitor = EntropyMonitor(floor_nats=0.15, drop_frac=0.6, window=8)
    probe_prompts = [m["messages"][0]["content"] for m in msgs[:8]]
    trainer = GKDTrainer(model=student, teacher_model=teacher, args=args,
                         processing_class=tok, train_dataset=ds,
                         callbacks=[RolloutWatch(tok, probe_prompts, monitor, arm["gen_T"])])
    trainer.train()
    trainer.save_model(f"../runs/lab07/{name}/final")
    RunManifest(name=name, config={**arm, **{"trainer": "GKDTrainer"}}, seed=SEED,
                artifacts_out={"checkpoint": f"../runs/lab07/{name}/final"},
                notes=monitor.report()).save(f"../runs/lab07/{name}")
    del student, teacher, trainer
    if device == "cuda":
        torch.cuda.empty_cache()
    return monitor

if RUN_TRAINING:
    reports = {}
    for name, arm in ARMS.items():
        print(f"\n=== arm: {name} {arm} ===")
        reports[name] = run_arm(name, arm, None).report()
    print(json.dumps(reports, indent=2))
else:
    print("RUN_TRAINING=False — Part B compiled but did not execute.")
    print("Watch the degenerate arm live: it is the whole point of running it.")

RUN_TRAINING=False — Part B compiled but did not execute.
Watch the degenerate arm live: it is the whole point of running it.


## Part C — The verdict

**Reading entropy trajectories** (the skill this lab exists to build):

- *Healthy on-policy:* entropy declines smoothly 10–30% over the run and plateaus. The decline
  is the student committing to what it can do; the plateau is the equilibrium with the
  teacher's spread. Samples stay varied in structure.
- *Early instability:* entropy jumps in the first ~50 steps on cold-started `on`. That is the
  sampled-ratio noise from Lab 00 §9 — student far from teacher, scores near-meaningless. It
  should settle. If it does not, warm-start.
- *Collapse:* an accelerating decline, usually late, usually after things looked fine. Samples
  develop tics (repeated openers, shrinking length) a hundred steps before the metric is
  obviously bad — which is why the callback prints text and not just numbers.

**Expected results.** `on` and `mixed` beat `off` on *generation-side* quality (their own
rollouts scored by the teacher) by a visible margin — that is exposure bias closing. `off` may
still match them on teacher-forced agreement; that is the point: teacher-forced metrics cannot
see the thing on-policy training fixes. `on-warm` reaches `on`'s quality in noticeably fewer
steps and with a calmer early trajectory — the sequencing argument made empirical. `degenerate`
must trip the monitor; if it survives 600 steps healthy, your thresholds are too loose to
protect a real run — tighten and rerun until the tripwire demonstrably works.

**Failure signatures** beyond collapse: rollouts that never emit EOS (see the Part A·3 mask
warning — lengthen the cap or fix the chat template); teacher-scoring OOM at long rollouts
(the teacher prefill batch grows with `max_new_tokens`; shrink generation batch, not training
batch); `on-warm` *worse* than `on` (your Lab 04 checkpoint is likely overfit to the corpus —
its narrowed distribution is a bad rollout policy; check its entropy against base).

**The verdict to write:** for your next real distillation, state your sequencing (how many
off-policy steps before switching on-policy), your `lmbda`/`beta`, and your monitor thresholds
— each justified by a number from *these* runs, not from the papers.

## Exercises

1. **`lmbda` fine-grain.** Add 0.25 and 0.75. GKD's paper reports task-dependent optima; find
   where the generation-quality curve bends on yours.
2. **β × on-policy interaction.** Rerun `on` with β=0 (forward KL on rollouts). Theory says
   this is the odd combination — punishing the student for teacher mass it never sampled.
   What actually happens to entropy?
3. **Monitor calibration curve.** Sweep `drop_frac` ∈ {0.3, 0.45, 0.6} over your five runs'
   logged trajectories (no retraining — replay the histories). Which setting catches
   `degenerate` earliest without false-firing on `on`?
4. **The buffer question.** GKDTrainer regenerates rollouts each step; older RL practice
   reuses a buffer for several epochs. Implement 2-epoch reuse and measure the staleness
   cost — this is the on-policy/off-policy boundary made continuous, and it previews the
   throughput problem Lab 08 solves properly.